# Notebook 17 — pytest — fixtures & parametrize patterns

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate → Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `16-numpy-embeddings-shape.ipynb`

**Next up:** `18-asyncio-queue-pipelines.ipynb`

---

## Learning objectives

- Mirror pytest ergonomics (`tmp_path`, parametrization) while staying notebook-friendly.
- Isolate filesystem tests via temporary directories.
- Catch expected failures with structured patterns.

## Table of contents

1. **Why migrate beyond bare asserts**
2. **`TemporaryDirectory` ≈ `tmp_path`**
3. **Table-driven ↔ `@pytest.mark.parametrize`**
4. **Progressive drills — raises → golden JSON → fixture reuse**
5. **Exercise — split_batches tests**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


### Dependency

```bash
pip install pytest
```

Notebook cells below run **without** pytest using stdlib **`tempfile`**; copy the collapsed solutions into **`tests/`** modules when you adopt pytest locally.


## 1 · Why migrate beyond bare asserts

*Explanation:* **`pytest`** discovers tests, captures stdout, reports diff snippets — migrate parsers (`JSON`, fences) early.


In [ ]:
# Bare notebook guard — pytest replaces loops like this with discovery + richer failures.


def split_batches(items: list[str], size: int) -> list[list[str]]:
    if size < 1:
        raise ValueError("size must be positive")
    return [items[i : i + size] for i in range(0, len(items), size)]


cases = [
    ([], 3, []),
    (["a"], 3, [["a"]]),
    (["a", "b", "c", "d"], 2, [["a", "b"], ["c", "d"]]),
]
for items, size, expected in cases:
    assert split_batches(items, size) == expected
print("table assertions OK")


## 2 · `TemporaryDirectory` ≈ `tmp_path`

*Explanation:* **`tmp_path`** is a pathlib **`Path`** — rehearsal uses **`TemporaryDirectory`** plus **`Path`** join semantics.


In [ ]:
from pathlib import Path
import json
import tempfile


def dump_roundtrip(payload: dict, folder: Path) -> dict:
    target = folder / "payload.json"
    target.write_text(json.dumps(payload))
    return json.loads(target.read_text())


with tempfile.TemporaryDirectory() as td:
    got = dump_roundtrip({"model": "mini"}, Path(td))
    assert got["model"] == "mini"
print("tmp mirror OK")


## 3 · Expect failures cleanly

*Explanation:* **`pytest.raises`** documents contracts — notebooks mimic **`try` / `except`** blocks until you promote snippets into **`tests/`**.


In [ ]:
def must_positive(x: int) -> int:
    if x <= 0:
        raise ValueError("need positive")
    return x


try:
    must_positive(0)
except ValueError as exc:
    print("caught:", exc)


---

## Progressive drills — **easy → harder**

**Golden-file parsers** deserve repeatable fixtures — drills mimic pytest ergonomics inside Jupyter.


### A · Easiest — inequality diagnostics

Prefer assertions that describe slices (`got[:40]`).


In [ ]:
text = '{"ok": true}'
assert '"ok"' in text
print("substring guard OK")


### B · Medium — normalize paths before comparison

Always **`resolve()`** when comparing tmp hierarchies.


In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "nested" / "file.txt"
    p.parent.mkdir(parents=True)
    p.write_text("x")
    assert p.resolve().parent.name == "nested"
print("path stable OK")


### C · Harder — record stderr-shaped payloads

Think regression snapshots for CLI tooling wrappers.


In [ ]:
payload = {"tool": "search", "argc": 3}
frozen_keys = tuple(sorted(payload))
assert frozen_keys == ("argc", "tool")
print("canonical tuple OK")


### Exercise — harden `split_batches`

1. Add **`assert`** checks (loop ok) proving **`split_batches`** rejects **`size <= 0`** with **`ValueError`**.
2. Add one positive-path assert with **`["x"] * 5`** and **`size=2`**.

<details><summary>Optional pytest module sketch</summary>

```python
import pytest
from lesson import split_batches

@pytest.mark.parametrize(
    "items,size,expected",
    [
        ([], 3, []),
        (["a", "b", "c", "d"], 2, [["a", "b"], ["c", "d"]]),
    ],
)
def test_batches_ok(items, size, expected):
    assert split_batches(items, size) == expected

def test_batches_bad_size():
    with pytest.raises(ValueError):
        split_batches(["a"], 0)
```

</details>


In [ ]:
# Uses split_batches from cell "Why migrate beyond bare asserts" — add checks below.


try:
    split_batches(["a"], 0)
except ValueError:
    print("reject non-positive sizes OK")
else:
    raise AssertionError("expected ValueError for size<=0")

assert split_batches(["x"] * 5, 2) == [["x", "x"], ["x", "x"], ["x"]]
print("batch shapes OK")


### Solution — split_batches checks

<details>
<summary>Click to expand</summary>

```python
def split_batches(items: list[str], size: int) -> list[list[str]]:
    if size < 1:
        raise ValueError("size must be positive")
    return [items[i : i + size] for i in range(0, len(items), size)]

try:
    split_batches(["a"], 0)
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

assert split_batches(["x"] * 5, 2) == [["x", "x"], ["x", "x"], ["x"]]
```

</details>
